# MLP surrogate for steel / SiC / UHMWPE armor
### Zhang *et al.* (2025) dataset — build, train, tune and validate a multi-layer perceptron (ReLU)

Predicts the ballistic performance of a 4-layer armor plate from its three layer thicknesses, using the 300-sample
Latin-hypercube dataset of Appendix A (*Composites Part B* **307**, 112961).

| | Inputs (mm) | Outputs |
|---|---|---|
| **Data** | `t1` front steel, `t2` SiC ceramic, `t3` UHMWPE laminate | `SEA` specific energy absorption (J/kg), `KER` projectile residual kinetic energy (J) |

**Model:** one single-output MLP per target · 3 inputs → `num_layers` × [Dense + ReLU + Dropout] → 1 linear output · Adam, MSE loss ·
Min-Max scaling · 5-fold cross-validation · MAE / RMSE / R² in physical units.

---
### Contents
**Part A — MLP surrogate** *(this is the project; run it top to bottom)*

| | Section | Paper item |
|---|---|---|
| [A0](#scrollTo=a0) | Setup and settings | Tables 7–8, row 1 |
| [A1](#scrollTo=a1) | Load the 300-sample dataset | §3.1, Appendix A |
| [A2](#scrollTo=a2) | Min-Max scaling and 5-fold split | §3.2, Eq. 17, Fig. 9 |
| [A3](#scrollTo=a3) | The MLP, training loop, metrics, CV driver | §3.3, Fig. 10, Eq. 18–19 |
| [A4](#scrollTo=a4) | Hyperparameter tuning table (optional, slow) | §3.4, Tables 7–8 |
| [A5](#scrollTo=a5) | SEA model: validation metrics and plots | §4.1, Figs. 12–13 |
| [A6](#scrollTo=a6) | KER model: validation metrics and plots | §4.2, Figs. 14–15 |
| [A7](#scrollTo=a7) | Final models and predictions for new inputs | — |

**Part B — SVM and Random Forest benchmark** *(separate study, run after A5 and A6; Part A never depends on it)*

| | Section | Paper item |
|---|---|---|
| [B1](#scrollTo=b1) | What the benchmark does and how it connects to Part A | §4.3 |
| [B2](#scrollTo=b2) | Imports, search grids and the nested-CV function | — |
| [B3](#scrollTo=b3) | Run SVM and RF (nested CV) | — |
| [B4](#scrollTo=b4) | Table 9: MLP vs SVM vs RF, paper vs reproduced | Table 9 |
| [B5](#scrollTo=b5) | Predicted-vs-simulated plots for all three models | Fig. 16 |

---
### How the cells depend on each other
| Cell | Needs | Creates |
|---|---|---|
| A0 | — | imports, `SEED`, `ACTIVATION`, `MLP_PARAMS`, `BATCH_SIZE`, `EPOCHS`, `RUN_HYPERPARAMETER_GRID` |
| A1 | A0 | `armor`, `X`, `y` |
| A2 | A1 | `scaler_X`, `scaler_Y`, `X_scale`, `y_scale`, `kf` |
| A3 | A0, A2 | `build_mlp`, `train_mlp`, `predict_scaled`, `regression_metrics`, `to_physical`, `run_mlp_cv`, `parity_plot`, `loss_plot` |
| A4 | A3 | `TABLE7_SEA`, `TABLE8_KER`, `grid_results` *(nothing later needs these)* |
| A5 | A3 | `mlp_sea`, `oof_pred_sea`, `sea_histories` |
| A6 | A3 | `mlp_ker`, `oof_pred_ker`, `ker_histories` |
| A7 | A3 | `final_models`, `predict_sea_ker` |
| B2–B5 | A1–A3, A5, A6 | `bench`, `oof_bench`, `table9` |

**Run time (Colab CPU):** A0–A3, A5–A7 take a few minutes in total. **A4 is the slow one (30–60 min)**: set
`RUN_HYPERPARAMETER_GRID = False` in A0 to skip it, nothing else depends on it. Part B adds a few more minutes.
**How to run:** *Runtime → Run all.* Nothing has to be uploaded: the 300 samples are embedded in A1.

---
## Part A — MLP surrogate

### A0 · Setup and settings
**What it does:** imports the libraries, fixes the random seed, and holds every setting in one place.

- `MLP_PARAMS` — the paper's best hyperparameters, one set per target (first row of Table 7 for SEA, Table 8 for KER).
- `BATCH_SIZE`, `EPOCHS`, optimizer details — **not reported in the paper**, so they are assumptions of this notebook.
- `RUN_HYPERPARAMETER_GRID` — switch for the slow tuning table in A4.

In [ ]:
#importing the packages
import os
import io
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility (seeds Python, NumPy and TensorFlow in one call)
SEED = 42
keras.utils.set_random_seed(SEED)

# ---------------------------------------------------------------------
# Activation function of every hidden layer.
# ACTIVATION and EPOCHS are the only two settings that differ between the ReLU and the sigmoid notebook.
# ---------------------------------------------------------------------
ACTIVATION = 'relu'      # paper, Section 3.3: "nonlinear activations--specifically, the ReLU function"

# Best MLP hyperparameters reported in the paper (first row of Table 7 and of Table 8)
MLP_PARAMS = {
    'SEA': dict(hidden_dim=128, num_layers=3, dropout=0.2, learning_rate=0.0002, weight_decay=0.001),   # Table 7
    'KER': dict(hidden_dim=64,  num_layers=3, dropout=0.2, learning_rate=0.0004, weight_decay=0.0005),  # Table 8
}

# ---------------------------------------------------------------------
# Settings the paper does NOT report (assumptions of this notebook):
#  * optimizer: Adam with Keras' decoupled weight decay (w <- w - lr*wd*w every step) and clipnorm=1.0,
#    both kept from the previous version of this notebook; Dropout after every hidden layer
#  * batch size and number of epochs (below). Figs. 12/14 of the paper end at about 71 (SEA) and 85 (KER) epochs,
#    which suggests early stopping on the validation fold; a fixed number of epochs is used here instead so that
#    the validation folds are never used to make training decisions
#  * the random 5-fold split and the seeds
# ---------------------------------------------------------------------
BATCH_SIZE = 12      # 12 divides every training-set size used here (204, 240, 300), so all batches are equal
EPOCHS = 300         # the training loss has flattened by then; 500 epochs gave no better CV result

# Re-run all 21 + 21 hyperparameter settings of Tables 7-8 with 5-fold CV (the slowest part of the notebook)
# Set to False to skip the tuning table and go straight to the two final MLPs (sections 6-7)
RUN_HYPERPARAMETER_GRID = True

sns.set_theme(style="ticks")
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 12
pd.set_option('display.width', 200)
tf.get_logger().setLevel('ERROR')    # hide tf.function retracing notices

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| hidden-layer activation:", ACTIVATION,
      "| batch size", BATCH_SIZE, "| epochs", EPOCHS)

### A1 · Load the 300-sample dataset
**What it does:** reads Appendix A (300 Latin-hypercube samples, embedded in the code cell below, so the cell is collapsed:
double-click it to see the code), forces numeric types, drops blanks, and splits the table into inputs `X` (t1, t2, t3) and outputs `y` (SEA, KER).

**Creates:** `armor` (DataFrame), `X` (300×3), `y` (300×2).

In [ ]:
#@title A1 · Embedded dataset (code hidden: double-click to expand) { display-mode: "form" }
# =====================================================================
# LOAD THE DATASET OF THE PAPER (Appendix A: 300 Latin-hypercube samples)
# =====================================================================
# If paper_dataset_LHS300.csv has been uploaded to Colab it is read from /content,
# otherwise the copy of Appendix A embedded below is used (identical values).
DATA_PATH = "/content/paper_dataset_LHS300.csv"

PAPER_DATA_CSV = """Sample,t1,t2,t3,SEA,KER
1,7.14,14.64,12,1632.16,211
2,3.52,7.45,17,1828.42,2270
3,6.4,11.56,33,1384.2,0.66
4,3.53,10.52,12,1772.64,2900
5,6.58,14.93,36,1357.33,0.91
6,4.07,10.44,25,1728.71,615
7,9.46,8.55,28,1334.31,31.7
8,8.52,8.33,37,1222.15,0.77
9,4.91,11.7,29,1582.92,87.4
10,5.34,12.24,40,1380.04,1.02
11,7.7,12.23,37,1283.62,1.27
12,4.61,6.49,26,1678.63,1410
13,5.15,8.43,16,1725.6,1770
14,5.65,9.21,30,1534.08,372
15,8.49,11.65,22,1449.48,0.82
16,10,10.94,18,1405.66,38.1
17,9.69,14.84,37,1209.74,0.56
18,5.56,12.6,22,1642.61,83
19,2.6,8.11,28,1820.23,1340
20,2.85,10.57,17,1968.87,1290
21,9.47,14.74,28,1293.6,0.94
22,3.57,13.31,29,1677.13,0.53
23,8.23,5.65,14,1511.92,1980
24,6.23,8.53,38,1331.03,87
25,7.52,13.54,32,1323.45,0.77
26,2.22,11.15,22,1975.99,657
27,7.29,7.95,35,1345.1,221
28,2.75,13.23,11,2065.15,1010
29,2.38,6.15,27,1861.92,1890
30,7.86,14.52,19,1494.27,0.49
31,3.89,14.48,36,1548.06,1.09
32,4.15,7.02,23,1750.32,1540
33,3.99,7.86,20,1795.87,1690
34,9.04,6.18,39,1174.92,0.34
35,7.63,10.74,32,1323.36,1.12
36,7.02,14.19,12,1638.56,348
37,5.6,14.57,38,1402.85,1.38
38,9.01,7.07,30,1332.72,305
39,6.98,10.23,35,1317.27,0.67
40,6.86,10.84,37,1313.58,1.33
41,4.51,10.39,31,1587.21,221
42,9.88,13.67,20,1357.21,0.69
43,6.02,11.34,38,1344.02,1.42
44,8.68,14.24,32,1290.29,1
45,7.5,14.88,33,1322.6,0.81
46,5.23,10.09,21,1752.54,882.05
47,4.85,11.79,19,1815.22,630.37
48,8.74,6.53,39,1460.39,0.28
49,6.09,13.43,36,1584.08,0.94
50,8.26,5.94,13,1532.39,1888.8
51,3.15,9.61,19,1938.33,1353.2
52,5.58,11.02,14,1753.53,1188
53,4.69,14.43,14,1913.87,108.29
54,8.43,12.98,13,1619.48,132.63
55,4.43,9.19,32,1762.49,425.03
56,2.59,7.68,29,1881.6,1438.9
57,2.32,10.12,29,1976.78,583.89
58,4.63,5.31,13,1700.92,3159
59,5.28,11.85,34,1681.04,1.18
60,3,12.08,21,1997,453.62
61,5.49,11.73,22,1769.68,215.41
62,8.14,9.56,12,1580.83,1201.3
63,8.85,7.98,38,1447.87,1.27
64,2.04,9.06,20,2002.28,1823.4
65,5.26,7.34,32,1682.01,743.45
66,8.81,11.28,17,1558.01,309.29
67,9.12,8.39,22,1501.51,556.65
68,2.44,15,35,1872.95,4.96
69,8.71,11.61,35,1444.84,0.62
70,9.26,7.2,29,1469.79,328.4
71,4.09,10.02,11,1746.36,2845.4
72,2.13,7.41,13,1903.98,3447.6
73,5.42,8.48,21,1516.64,2997.1
74,6.14,6.76,10,1625.35,2750.6
75,2.1,5.36,29,1484.83,5137.3
76,9.35,13.3,21,1497.82,0.5
77,6.51,13.04,27,1641.28,0.78
78,5.7,10.48,38,1627.01,1.39
79,5.02,7.53,27,1723.86,1016.4
80,8.61,12.69,31,1470.7,0.85
81,9.9,8.71,25,1469.78,85.77
82,5.05,9.95,25,1747.34,683.94
83,6.79,12.28,29,1610.88,1.32
84,9.38,8.57,21,1504.79,413.86
85,6.33,7,30,1642.73,545.34
86,6.7,9.7,14,1668.16,1239.9
87,4.78,13.37,25,1799.11,0.45
88,8.18,8.02,35,1511.32,0.6
89,7.82,14.33,28,1527.36,1.32
90,6,10.26,34,1645.46,0.2
91,6.08,8.16,16,1675.57,1631.5
92,8.37,13.65,34,1453.11,1.1
93,7.11,6.05,34,1574.48,390.65
94,3.09,6.65,35,1809.44,1108.8
95,9.73,11.3,17,1520.13,66.54
96,5.83,13.08,13,1783.36,571.14
97,8.12,5.02,13,1526.49,2140.6
98,7.59,10,28,1582.82,64.91
99,3.93,13.18,23,1906.86,0.5
100,2.64,8.82,23,1909.64,1679.3
101,8.48,12.34,24,1538.45,0.58
102,7.98,7.49,23,1550.21,851.85
103,2.83,8.98,19,1921.24,1847.4
104,9.43,6.44,32,1453.79,218.57
105,6.63,12.32,12,1690.84,966.65
106,7.95,5.85,15,1541.42,1851.1
107,5.38,13.86,31,1678.3,0.92
108,5.67,13.76,32,1647.59,0.72
109,4.3,13.2,35,1732.34,0.68
110,5.32,6.36,31,1664.84,1069.6
111,3.59,14.38,24,1908.93,0.63
112,4.2,8.25,16,1817.65,1914.7
113,7.19,6.28,25,1589.5,979.99
114,5.5,8.6,10,1720.07,2231.5
115,3.74,5.38,21,1800.09,2276.5
116,7.48,10.98,37,1509.53,1.02
117,5.76,6.59,27,1655.03,1186
118,5.4,5.47,18,1678.02,2252.4
119,3.38,10.4,38,1778.75,349.38
120,9.57,13,15,1535.77,0.46
121,9.79,14.69,12,1529,0.34
122,2.15,10.15,11,2036.71,2371.5
123,3.32,9.41,37,1817.41,285.94
124,6.73,7.13,24,1618.29,1032.5
125,5.84,13.44,39,1574.66,1.19
126,9.32,14.21,17,1522.9,0.48
127,2.53,5.76,18,1856.89,3115.2
128,3.95,8.19,22,1820.09,1478.7
129,7.94,10.07,38,1482.15,0.73
130,3.03,12.03,12,2000.64,1442.1
131,8.77,6.08,17,1488.98,1596.7
132,2.26,13.6,33,1939.33,3.47
133,4.02,10.19,31,1800.54,389.75
134,2.47,9.47,39,1884.33,192.26
135,5.9,6.85,26,1660.68,1094.9
136,8.41,13.9,15,1600.54,45.88
137,8.86,11.94,22,1534.5,1.05
138,9.16,13.36,22,1501.06,0.56
139,3.49,7.56,38,1784.71,570.48
140,9.29,6.67,35,1456.85,0.75
141,4.05,14.08,30,1796.88,1.14
142,6.68,5.13,22,1596.9,1682.1
143,4.7,9.66,39,1703.91,0.31
144,3.13,8.45,11,1883.64,2815
145,9.08,9.27,24,1509.87,227.32
146,8.31,8.3,18,1551.87,962.23
147,9.75,6.38,30,1450.18,223.26
148,7.8,5.79,22,1549.06,1266.2
149,6.55,6.42,18,1633.39,1670
150,5.18,5.26,18,1686.75,2371.6
151,4.37,11.06,15,1865.88,1123.9
152,7.05,9.88,25,1629.41,279.11
153,5.94,10.87,11,1724.98,1471
154,7.39,8.9,12,1593.78,1690.3
155,6.51,5.53,24,1627.62,1291.9
156,3.41,9.83,28,1603.77,2882
157,3.06,12.45,23,1971.82,320.36
158,8.2,9.25,15,1572.31,1003.3
159,9.02,13.81,26,1473.85,0.96
160,3.2,11.47,37,1830.36,1.07
161,6.76,7.71,20,1459.66,2781.1
162,5.22,12.41,36,1659.09,0.65
163,9.18,12.95,20,1520.09,0.43
164,6.04,10.71,14,1717.07,1184.6
165,5.52,5.92,30,1669.6,1024.9
166,8.1,14.34,11,1638.27,217.76
167,3.34,5.08,16,1788.55,3241.1
168,5.44,14.95,19,1786.47,0.38
169,2.35,11.08,34,1958.92,0.43
170,7.68,9.02,15,1585.73,1269.6
171,9.62,13.58,23,1463.24,0.51
172,5.72,5.71,31,1652.97,957.75
173,8.63,7.75,34,1493.22,8.32
174,4.28,11.53,19,1876.84,615.39
175,5.89,12.13,16,1758.27,560.18
176,7.75,12.72,37,1475.82,1.34
177,6.47,7.67,27,1635.52,715.58
178,9.5,10.64,39,1381.46,1.03
179,3.8,10.54,34,1823.97,2.69
180,2.19,12.37,31,1995.47,1.15
181,2.7,5.46,17,1829.4,3314.7
182,8.79,10.36,24,1538.81,0.55
183,7.07,7.19,16,1598.44,1704.8
184,5.81,8.09,24,1685.69,979.31
185,4.57,6.22,31,1725.59,1114.1
186,3.3,5.04,26,1812.99,2056.7
187,9.96,8.79,20,1436.82,815.21
188,3.82,5.1,20,1561.49,4195
189,6.36,10.92,40,1557.89,1.61
190,4.88,7.81,24,1747.26,1186.3
191,2.78,12.12,11,2054.65,1378
192,5.01,5.19,18,1706.49,2348
193,9.2,14.55,39,1364.67,0.88
194,8.92,13.8,18,1544.82,0.42
195,7.9,11.44,27,1560.85,1.44
196,4.96,8.87,36,1706.44,168.12
197,7.31,6.02,36,1549.76,325.31
198,5.76,13.11,39,1583.81,1.1
199,3.64,13.99,40,1723.51,0.8
200,6.25,12.52,36,1583.04,0.86
201,6.66,7.79,16,1630.3,1645
202,9.59,10.77,23,1491.99,1.12
203,6.39,10.62,36,1594.33,1.4
204,8,14.16,33,1477.86,1.05
205,4.23,9.38,22,1838.68,939.85
206,6.92,7.59,21,1609.5,1203
207,2.87,11.92,39,1830.51,1.33
208,5.08,7.12,17,1734.53,1925.2
209,2.3,5.3,16,1835.14,3730.6
210,6.2,14.12,32,1604.67,0.73
211,9.25,8.67,16,1404.39,1947.5
212,4.65,5.61,21,1715.12,2207.1
213,2.07,7.33,13,1913.76,3444.3
214,6.44,7.87,14,1647.72,1832.1
215,7.15,14.61,23,1613.7,0.45
216,6.6,12.85,14,1714.79,510.64
217,7.28,12.65,18,1651.56,229.87
218,3.24,9.58,26,1596.06,3280.6
219,8.55,14.45,32,1450.17,0.77
220,4.81,6.72,29,1732.69,1002.6
221,4.39,14.27,28,1786.84,0.6
222,5.97,5.56,12,1603.97,3003.8
223,8.07,9.12,26,1556.86,272.17
224,7.01,10.32,28,1626.95,0.15
225,7.55,13.17,36,1491.43,1.31
226,8.94,7.26,40,1434.63,0.63
227,4.17,7.6,30,1765.88,1005.2
228,6.16,11.1,14,1640.18,1725.4
229,2.91,5.7,33,1850.25,1255.6
230,7.24,12.82,36,1514.27,1.12
231,4.88,12.57,25,1801.57,0.43
232,6.27,9.92,13,1689.59,1444.1
233,5.16,12.87,25,1772.65,0.39
234,4.32,9.84,26,1808.59,641.23
235,7.21,13.93,29,1563.41,0.82
236,2.01,8.32,34,1909.27,1081.2
237,7.87,9.46,11,1589.71,1412.5
238,8.05,6.91,29,1532.92,509.3
239,3.45,9.32,19,1685.12,3108.2
240,4.55,5.97,34,1717.68,932.85
241,3.9,12.57,11,1959.83,1015.5
242,7.43,5.82,18,1558.73,1787.5
243,2.17,12.74,37,1910.27,1.65
244,7.76,10.68,17,1615.88,548.5
245,8.56,6.78,19,1513.73,1245.6
246,2.67,9.09,24,1916.86,1450.9
247,3.71,12.2,27,1887.53,49.38
248,4.26,5.87,34,1737.56,994.27
249,9.85,8.9,20,1491.35,291.07
250,4.76,10.28,31,1739.83,335.22
251,3.42,5.57,32,1810.71,1282.5
252,3.78,7.91,15,1817.65,2406.6
253,4.42,14.71,23,1835.9,0.45
254,8.89,6.33,40,1446.02,0.21
255,9.68,7.28,16,1471.62,1124.4
256,8.97,12,38,1405.15,1.09
257,8.27,11.88,33,1483.48,1.32
258,3.22,8.05,30,1863.89,894.14
259,3.09,6.94,20,1846.46,2355.8
260,3.84,14.81,28,1828.75,0.53
261,6.84,6.51,23,1610.34,1191.9
262,6.3,13.7,14,1762.64,201.49
263,7.45,14.04,24,1589.75,0.81
264,8.38,9.8,14,1525.04,1366.4
265,2.53,12.78,12,2105.2,1027.7
266,7.61,9.36,33,1551.07,0.13
267,8.67,9.72,26,1536.53,0.76
268,7.71,11.74,16,1636.49,355.01
269,6.9,11.57,35,1557.77,1.04
270,7.38,9.76,25,1597.36,370.4
271,8.32,6.11,10,1500.64,2387.1
272,7.34,8.64,30,1570.3,341.33
273,3.61,14.02,39,1736.19,0.91
274,9.4,8.23,15,1504.2,962.92
275,3.66,10.83,10,1825.24,2591.5
276,9.76,7.04,21,1465.01,742.94
277,9.12,11.81,31,1448.89,1.19
278,6.82,6.27,27,1595.08,1008.9
279,3.68,6.82,11,1760.08,3467.2
280,6.11,12.04,37,1589.17,0.89
281,9.64,6.9,15,1458.08,1413.5
282,5.11,12.5,26,1770.96,0.57
283,4.46,13.47,25,1826.73,0.41
284,4.48,11.24,33,1765.23,0.13
285,2.95,11.4,27,1934.72,415.46
286,2.73,11.42,17,2030.94,995.21
287,9.52,7.38,13,1486.1,1328.5
288,6.95,5.42,26,1587.29,1178.6
289,4.73,11.21,28,1788.12,107.3
290,2.65,5.21,19,1829.64,3176
291,2.92,13.53,27,1952.2,1.1
292,9.92,12.9,13,1526.97,40.36
293,4.13,9.16,20,1836.56,1280.5
294,9.83,9.53,11,1462.37,1306.7
295,2.41,11.17,19,2060.54,868.43
296,2.27,8.95,33,1919.39,800.52
297,4.97,8.74,38,1694.95,89.38
298,2.48,14.77,33,1898.33,0.58
299,3.27,6.61,30,1833.42,1293
300,2.97,13.92,35,1839.36,0.64
"""

if os.path.exists(DATA_PATH):
    armor = pd.read_csv(DATA_PATH)
    print("Loaded", DATA_PATH)
else:
    armor = pd.read_csv(io.StringIO(PAPER_DATA_CSV))
    print("Loaded the embedded copy of Appendix A")

armor = armor.set_index('Sample')
armor.head()

In [ ]:
# The first 3 variables are inputs: t1 (front steel), t2 (SiC ceramic), t3 (UHMWPE) thickness in mm
X = armor[['t1', 't2', 't3']].values

# The last 2 variables are outputs: SEA (J/kg) and KER (J)
y = armor[['SEA', 'KER']].values

# Use numpy slicing [:5] to see the first 5 rows safely
print("Input (X) head (first 5 rows):")
print(X[:5])

print("\nOutput (y) head (first 5 rows):")
print(y[:5])

In [ ]:
# 1. Force all data columns to be strictly numeric (coerces hidden text or corrupt values to NaN)
for col in armor.columns:
    armor[col] = pd.to_numeric(armor[col], errors='coerce')

# 2. Drop any row containing any NaN or blank cell completely
armor = armor.dropna()

# 3. Extract your clean raw NumPy arrays
X = armor[['t1', 't2', 't3']].values  # t1, t2, t3
y = armor[['SEA', 'KER']].values      # SEA, KER

print("Cleaned Dataset Shape:", armor.shape)
print("Any NaNs left in X?", np.isnan(X).any())
print("Any NaNs left in y?", np.isnan(y).any())
assert armor.shape == (300, 5), "Appendix A contains 300 samples of t1, t2, t3, SEA, KER"

### A2 · Min-Max scaling and 5-fold split
**What it does:** squeezes every input and output into [0, 1] (Eq. 17) so thickness in mm and energy in thousands of joules are on comparable scales, and defines the 5-fold splitter used everywhere
(each fold: 240 training / 60 validation samples).

**Creates:** `scaler_X`, `scaler_Y` (kept to convert predictions back to physical units), `X_scale`, `y_scale`, `kf`.

In [ ]:
# 1. Min-Max scale every input and output to [0, 1] (Eq. 17)
scaler_X = MinMaxScaler()
scaler_Y = MinMaxScaler()

X_scale = scaler_X.fit_transform(X)
y_scale = scaler_Y.fit_transform(y)

# Isolate the 2 targets cleanly based on the dataset columns
y_scale_sea = y_scale[:, 0]   # Output 1: SEA (J/kg)
y_scale_ker = y_scale[:, 1]   # Output 2: KER (J)

# 2. Set up the 5-Fold Cross-Validator as specified in Section 3.2 (each fold: 80 % training / 20 % validation)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_scale), 1):
    print(f"Fold {fold}: {len(train_idx)} training samples / {len(test_idx)} validation samples")

### A3 · The MLP, training loop, metrics and CV driver
**What it does:** defines the building blocks only (nothing is trained yet).

| Function | Purpose |
|---|---|
| `build_mlp` | Fig. 10 network: 3 inputs → `num_layers` × [Dense + ReLU + Dropout] → 1 linear output; Adam with weight decay, MSE loss |
| `train_mlp` | mini-batch training with a fresh shuffle every epoch, compiled into one TensorFlow loop for speed; returns per-epoch training/validation loss |
| `predict_scaled` | prediction with dropout off |
| `regression_metrics` | MAE, RMSE (root of Eq. 18) and R² (Eq. 19) in physical units |
| `to_physical` | undoes the Min-Max scaling of one output column (0 = SEA, 1 = KER) |
| `run_mlp_cv` | 5-fold CV of one MLP: fresh model per fold, validation metrics, out-of-fold predictions, loss histories |
| `parity_plot`, `loss_plot` | predicted-vs-simulated plot (Figs. 13/15/16) and loss curves (Figs. 12/14) |

> One network per target: the paper's text says one network with two outputs, but its Fig. 10 shows one output node and Tables 7–8 give a different best setting for SEA and KER.
> Keras applies `weight_decay` in decoupled form, a weak regulariser at these small values; the paper does not say which form it used.

In [ ]:
def build_mlp(hidden_dim, num_layers, dropout, learning_rate, weight_decay, activation=ACTIVATION):
    """Fig. 10: 3 inputs -> num_layers x [Dense(hidden_dim) + Dropout] -> 1 linear output."""
    model = Sequential([Input(shape=(3,))])
    for _ in range(num_layers):
        model.add(Dense(hidden_dim, activation=activation))
        model.add(Dropout(dropout))
    model.add(Dense(1, activation='linear'))
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate, weight_decay=weight_decay, clipnorm=1.0)
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model


def train_mlp(model, X_train, y_train, X_val=None, y_val=None, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """Mini-batch training with a fresh shuffle every epoch -- the same procedure as
    model.fit(X_train, y_train, batch_size, epochs, shuffle=True, validation_data=(X_val, y_val)),
    including Keras' smaller last batch, but with all epochs inside one compiled TensorFlow graph.
    On this 300-sample dataset model.fit spends most of its time on per-epoch overhead, which would make the
    42 x 5 fits of Tables 7-8 take hours. Returns a Keras-style history dict (per-epoch training / validation MSE)."""
    has_val = X_val is not None
    Xt = tf.constant(X_train, tf.float32)
    yt = tf.constant(np.reshape(y_train, (-1, 1)), tf.float32)
    Xv = tf.constant(X_val if has_val else X_train[:1], tf.float32)
    yv = tf.constant(np.reshape(y_val if has_val else y_train[:1], (-1, 1)), tf.float32)
    n = len(X_train)
    n_batches = -(-n // batch_size)              # ceil(n / batch_size); the last batch may be smaller
    optimizer = model.optimizer
    optimizer.build(model.trainable_variables)
    mse = keras.losses.MeanSquaredError()

    @tf.function
    def run_epochs():
        train_loss = tf.TensorArray(tf.float32, size=epochs)
        val_loss = tf.TensorArray(tf.float32, size=epochs)
        for epoch in tf.range(epochs):
            order = tf.random.shuffle(tf.range(n))
            total = tf.constant(0.0)
            for b in tf.range(n_batches):
                idx = order[b * batch_size:tf.minimum((b + 1) * batch_size, n)]
                xb, yb = tf.gather(Xt, idx), tf.gather(yt, idx)
                with tf.GradientTape() as tape:
                    loss = mse(yb, model(xb, training=True))          # dropout is active while training
                grads = tape.gradient(loss, model.trainable_variables)
                optimizer.apply_gradients(zip(grads, model.trainable_variables))   # Adam + weight decay + clipnorm
                total += loss * tf.cast(tf.shape(idx)[0], tf.float32)
            train_loss = train_loss.write(epoch, total / n)                # sample-weighted, as Keras reports it
            val_loss = val_loss.write(epoch, mse(yv, model(Xv, training=False)))
        return train_loss.stack(), val_loss.stack()

    train_hist, val_hist = run_epochs()
    history = {'loss': train_hist.numpy()}
    if has_val:
        history['val_loss'] = val_hist.numpy()
    return history


def predict_scaled(model, X_in):
    """Scaled prediction of a trained MLP (dropout off)."""
    return model(tf.constant(X_in, tf.float32), training=False).numpy().flatten()


def regression_metrics(y_true, y_pred):
    """MAE, RMSE (square root of Eq. 18) and R2 (Eq. 19) in physical units."""
    return dict(MAE=mean_absolute_error(y_true, y_pred),
                RMSE=np.sqrt(mean_squared_error(y_true, y_pred)),
                R2=r2_score(y_true, y_pred))


def to_physical(values_scaled, col):
    """Inverse Min-Max transform of one output column (0 = SEA, 1 = KER)."""
    # scaler_Y expects 2 columns: [SEA, KER]
    dummy = np.zeros((len(values_scaled), 2))
    dummy[:, col] = values_scaled
    return scaler_Y.inverse_transform(dummy)[:, col]


def run_mlp_cv(col, params, rows=None, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True):
    """5-fold CV of one MLP on the selected rows (default: all 300 samples).
    Returns per-fold metrics (validation and training), out-of-fold predictions and loss histories."""
    rows = np.arange(len(X_scale)) if rows is None else np.asarray(rows)
    X_rows, y_rows_scaled, y_rows_raw = X_scale[rows], y_scale[rows, col], y[rows, col]
    fold_metrics, histories = [], []
    oof_pred = np.zeros(len(rows))
    for fold, (train_idx, test_idx) in enumerate(kf.split(X_rows), 1):
        # Reset the backend and re-seed so that every fold (and every grid setting) is reproducible
        keras.backend.clear_session()
        keras.utils.set_random_seed(SEED + fold)
        model = build_mlp(**params)
        history = train_mlp(model, X_rows[train_idx], y_rows_scaled[train_idx],
                            X_rows[test_idx], y_rows_scaled[test_idx], epochs=epochs, batch_size=batch_size)
        history['target_var'] = np.var(y_rows_scaled[train_idx])
        pred_raw = to_physical(predict_scaled(model, X_rows[test_idx]), col)
        train_raw = to_physical(predict_scaled(model, X_rows[train_idx]), col)
        m = regression_metrics(y_rows_raw[test_idx], pred_raw)
        m_train = regression_metrics(y_rows_raw[train_idx], train_raw)
        m.update(R2_train=m_train['R2'], RMSE_train=m_train['RMSE'])
        fold_metrics.append(m)
        histories.append(history)
        oof_pred[test_idx] = pred_raw
        if verbose:
            print(f"Fold {fold} Complete. R² Score: {m['R2']:.4f} | MAE: {m['MAE']:.2f} | RMSE: {m['RMSE']:.2f} "
                  f"| training R²: {m['R2_train']:.4f}")
    return pd.DataFrame(fold_metrics), oof_pred, histories


def parity_plot(ax, true_raw, pred_raw, label, box_color, unit_name):
    """Predicted-vs-simulated plot in the style of Figs. 13, 15 and 16. The R2 shown is that of the plotted points.
    The paper plots one 20 % validation set (60 points); here all 300 out-of-fold predictions are plotted."""
    r2_points = r2_score(true_raw, pred_raw)
    ax.scatter(true_raw, pred_raw, color='black', edgecolor='none', s=25, label='Out-of-fold predictions (5 folds)')
    lo = min(true_raw.min(), pred_raw.min()); hi = max(true_raw.max(), pred_raw.max())
    pad = 0.05 * (hi - lo); lo, hi = lo - pad, hi + pad
    ax.plot([lo, hi], [lo, hi], color='red', linestyle='--', linewidth=1.5)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel(f'Simulation {unit_name}'); ax.set_ylabel(f'Predicted {unit_name}')
    ax.text(0.68, 0.90, f'$R^2$ = {r2_points:.2f}', transform=ax.transAxes, color='red', fontsize=13, fontweight='bold')
    ax.text(0.12, 0.75, label, transform=ax.transAxes, fontsize=14, fontweight='bold',
            bbox=dict(facecolor=box_color, alpha=0.8, edgecolor='none', pad=6))
    ax.legend(loc='upper left', frameon=False, fontsize=10)


def loss_plot(histories, title, paper_note):
    """Training / validation MSE per epoch, averaged over the 5 folds (Figs. 12 and 14).
    The MSE is divided by the variance of the scaled training target, which puts it on the scale of a z-scored
    target -- the scale of the paper's Figs. 12/14 (start near 1, validation level near 1 - R2)."""
    train = np.mean([h['loss'] / h['target_var'] for h in histories], axis=0)
    val = np.mean([h['val_loss'] / h['target_var'] for h in histories], axis=0)
    plt.figure(figsize=(6.5, 4.5))
    plt.plot(train, label='Training Loss')
    plt.plot(val, label='Validation Loss')
    plt.xlabel('Epoch'); plt.ylabel('MSE loss / variance of training target')
    plt.title(title); plt.legend(frameon=False)
    plt.text(0.35, 0.80, paper_note, transform=plt.gca().transAxes, fontsize=9, color='gray')
    plt.tight_layout(); plt.show()
    print(f"Loss on the paper's scale: validation {val[9]:.3f} at epoch 10, {val[-1]:.3f} at epoch {len(val)} "
          f"| training {train[-1]:.3f} at epoch {len(train)}")

### A4 · Hyperparameter tuning table (optional, slow)
**What it does:** re-trains all 21 SEA settings (Table 7) and 21 KER settings (Table 8) with the same 5-fold CV and prints the paper's MAE / RMSE / R² next to the reproduced ones,
the Spearman rank correlation between the two R² columns, and the rank of the paper's selected setting (row 1).

> **Takes 30–60 min.** Set `RUN_HYPERPARAMETER_GRID = False` in A0 to skip it. A5–A7 do not need it.

In [ ]:
# Hidden_dim, Num_layers, Dropout, Learning rate, Weight_decay, paper MAE, paper RMSE, paper R2
TABLE7_SEA = [
    (128, 3, 0.2,  0.0002,  0.001,  35.53, 56.62, 0.901),
    (64,  2, 0.1,  0.001,   0.01,   38.21, 62.05, 0.872),
    (96,  3, 0.0,  0.0003,  0.0001, 37.95, 60.83, 0.884),
    (128, 1, 0.3,  0.0005,  0.1,    40.33, 64.81, 0.832),
    (80,  2, 0.4,  0.01,    0.001,  45.67, 69.28, 0.781),
    (128, 5, 0.1,  0.0002,  0.0005, 42.18, 67.08, 0.815),
    (72,  3, 0.0,  0.0003,  0.0001, 36.89, 60.00, 0.889),
    (112, 2, 0.3,  0.00005, 0.01,   39.77, 64.12, 0.846),
    (128, 3, 0.5,  0.0002,  0.001,  41.02, 66.33, 0.829),
    (96,  4, 0.2,  0.0001,  0.1,    44.56, 70.00, 0.793),
    (256, 3, 0.1,  0.001,   0.001,  47.83, 72.16, 0.751),
    (64,  1, 0.0,  0.0002,  0.0,    37.45, 61.24, 0.880),
    (128, 2, 0.2,  0.00001, 0.001,  38.77, 62.85, 0.868),
    (144, 4, 0.3,  0.0005,  0.0,    43.91, 68.56, 0.802),
    (100, 2, 0.4,  0.00015, 0.05,   42.65, 67.83, 0.818),
    (128, 3, 0.1,  0.0003,  0.01,   39.21, 63.25, 0.856),
    (88,  3, 0.2,  0.0004,  0.005,  41.88, 65.97, 0.826),
    (200, 5, 0.0,  0.0002,  0.001,  46.52, 71.42, 0.772),
    (128, 3, 0.2,  0.1,     0.001,  52.34, 76.16, 0.682),
    (96,  2, 0.25, 0.00008, 0.1,    44.12, 69.65, 0.798),
    (64,  4, 0.35, 0.00025, 0.02,   48.76, 72.81, 0.738),
]
TABLE8_KER = [
    (64, 3, 0.2,  0.0004,  0.0005,  109.66, 190.88, 0.950),
    (48, 2, 0.1,  0.0008,  0.0001,  115.15, 199.50, 0.931),
    (64, 2, 0.0,  0.0004,  0.0,     112.89, 195.43, 0.938),
    (80, 4, 0.3,  0.001,   0.001,   125.28, 216.79, 0.898),
    (64, 3, 0.0,  0.00005, 0.0005,  110.41, 193.65, 0.942),
    (56, 3, 0.4,  0.0004,  0.01,    121.33, 207.34, 0.913),
    (64, 3, 0.2,  0.1,     0.0005,  145.17, 240.83, 0.828),
    (72, 4, 0.15, 0.00005, 0.005,   118.94, 202.49, 0.921),
    (64, 3, 0.05, 0.0008,  0.0,     112.25, 194.94, 0.935),
    (40, 3, 0.1,  0.0004,  0.0005,  130.61, 219.09, 0.892),
    (96, 4, 0.1,  0.0004,  0.0001,  120.87, 204.94, 0.908),
    (64, 4, 0.25, 0.001,   0.0005,  125.56, 214.48, 0.894),
    (64, 3, 0.1,  0.0006,  0.1,     135.32, 225.82, 0.876),
    (48, 5, 0.35, 0.0004,  0.001,   132.77, 221.58, 0.883),
    (64, 3, 0.1,  0.0005,  0.0,     110.98, 192.36, 0.938),
    (88, 3, 0.2,  0.0008,  0.00001, 116.45, 200.01, 0.922),
    (52, 4, 0.4,  0.00008, 0.02,    123.21, 212.14, 0.901),
    (64, 2, 0.0,  0.01,    0.0005,  140.63, 234.57, 0.852),
    (80, 3, 0.1,  0.0004,  0.0001,  111.12, 193.65, 0.937),
    (64, 5, 0.3,  0.0002,  0.001,   121.05, 209.77, 0.909),
    (56, 2, 0.3,  0.0003,  0.002,   118.88, 204.94, 0.918),
]

grid_results = {}
if RUN_HYPERPARAMETER_GRID:
    for name, col, table in [('SEA', 0, TABLE7_SEA), ('KER', 1, TABLE8_KER)]:
        print(f"=== Re-running Table {7 if name == 'SEA' else 8} ({name}): {len(table)} settings x 5 folds ===")
        rows = []
        for i, (h, nl, dr, lr, wd, mae_p, rmse_p, r2_p) in enumerate(table, 1):
            params = dict(hidden_dim=h, num_layers=nl, dropout=dr, learning_rate=lr, weight_decay=wd)
            fm, _, _ = run_mlp_cv(col, params, verbose=False)
            mean = fm.mean()
            rows.append(dict(Row=i, Hidden_dim=h, Num_layers=nl, Dropout=dr, Learning_rate=lr, Weight_decay=wd,
                             MAE_paper=mae_p, MAE_repr=mean['MAE'], RMSE_paper=rmse_p, RMSE_repr=mean['RMSE'],
                             R2_paper=r2_p, R2_repr=mean['R2']))
            print(f"  row {i:2d} done: R2 paper {r2_p:.3f} | reproduced {mean['R2']:.3f}")
        df = pd.DataFrame(rows).set_index('Row')
        df['Rank_paper'] = df['R2_paper'].rank(ascending=False, method='min').astype(int)
        df['Rank_repr'] = df['R2_repr'].rank(ascending=False, method='min').astype(int)
        rho = spearmanr(df['R2_paper'], df['R2_repr']).correlation
        grid_results[name] = dict(table=df, spearman=rho, rank_of_paper_best=int(df.loc[1, 'Rank_repr']),
                                  best_row=int(df['R2_repr'].idxmax()))
        print(df.round(4).to_string())
        print(f"Spearman rank correlation (paper R2 vs reproduced R2): {rho:.3f}")
        print(f"Paper's selected setting (row 1) ranks {grid_results[name]['rank_of_paper_best']} of {len(df)} "
              f"in the reproduction; best reproduced setting = row {grid_results[name]['best_row']}\n")
else:
    print("RUN_HYPERPARAMETER_GRID = False -> Tables 7-8 skipped; the paper's best settings are used below.")

### A5 · SEA model: validation metrics and plots
**What it does:** 5-fold CV of the SEA network with its Table 7 setting
(`hidden_dim=128, num_layers=3, dropout=0.2, learning_rate=0.0002, weight_decay=0.001`), then prints the mean validation R², MAE and RMSE against the paper
(**R² 0.901, MAE 35.53 J/kg, RMSE 56.62 J/kg**) and draws the loss curve (Fig. 12) and the predicted-vs-simulated plot (Fig. 13).

**Creates:** `mlp_sea`, `oof_pred_sea`, `sea_histories`.

In [ ]:
# =====================================================================
# SEA MODEL: 5-FOLD CROSS-VALIDATION, LOSS CURVES AND PARITY PLOT
# =====================================================================
print("=== Starting 5-Fold Cross-Validation & Visual Tracking for SEA Model ===")
sea_fold_metrics, oof_pred_sea, sea_histories = run_mlp_cv(0, MLP_PARAMS['SEA'])
mlp_sea = sea_fold_metrics.mean()

print("\n==============================================")
print("      FINAL AGGREGATED 5-FOLD SEA METRICS      ")
print("==============================================")
print(f"FINAL MEAN R² SCORE : {mlp_sea['R2']:.4f}   (paper: 0.901)")
print(f"FINAL MEAN MAE VALUE: {mlp_sea['MAE']:.2f} J/kg   (paper: 35.53)")
print(f"FINAL MEAN RMSE     : {mlp_sea['RMSE']:.2f} J/kg   (paper: 56.62)")
print(f"R² of all 300 out-of-fold predictions together: {r2_score(y[:, 0], oof_pred_sea):.4f}")
print(f"Mean training-fold R²: {mlp_sea['R2_train']:.4f}")
print("==============================================")

# Fig. 12: MSE loss convergence
loss_plot(sea_histories, f'Fig. 12 reproduced: SEA MLP loss ({ACTIVATION})',
          'paper Fig. 12: ends at ~71 epochs,\nvalidation loss levels off at ~0.10')

# Fig. 13: predicted vs simulated SEA (every sample predicted by the fold model that did not see it)
fig, ax = plt.subplots(figsize=(6, 5.5))
parity_plot(ax, y[:, 0], oof_pred_sea, 'MLP-SEA', '#8cc63f', 'specific energy absorption (J/kg)')
plt.tight_layout(); plt.show()

### A6 · KER model: validation metrics and plots
**What it does:** the same procedure for the residual-kinetic-energy network with its Table 8 setting
(`hidden_dim=64, num_layers=3, dropout=0.2, learning_rate=0.0004, weight_decay=0.0005`), compared with the paper
(**R² 0.95, MAE 109.66 J, RMSE 190.88 J**); loss curve (Fig. 14) and parity plot (Fig. 15).

**Creates:** `mlp_ker`, `oof_pred_ker`, `ker_histories`.

In [ ]:
# =====================================================================
# KER MODEL: 5-FOLD CROSS-VALIDATION, LOSS CURVES AND PARITY PLOT
# =====================================================================
print("=== Starting 5-Fold Cross-Validation & Visual Tracking for KER Model ===")
ker_fold_metrics, oof_pred_ker, ker_histories = run_mlp_cv(1, MLP_PARAMS['KER'])
mlp_ker = ker_fold_metrics.mean()

print("\n==============================================")
print("      FINAL AGGREGATED 5-FOLD KER METRICS      ")
print("==============================================")
print(f"FINAL MEAN R² SCORE : {mlp_ker['R2']:.4f}   (paper: 0.95)")
print(f"FINAL MEAN MAE VALUE: {mlp_ker['MAE']:.2f} J   (paper: 109.66)")
print(f"FINAL MEAN RMSE     : {mlp_ker['RMSE']:.2f} J   (paper: 190.88)")
print(f"R² of all 300 out-of-fold predictions together: {r2_score(y[:, 1], oof_pred_ker):.4f}")
print(f"Mean training-fold R²: {mlp_ker['R2_train']:.4f}")
print("==============================================")

# Fig. 14: MSE loss convergence
loss_plot(ker_histories, f'Fig. 14 reproduced: KER MLP loss ({ACTIVATION})',
          'paper Fig. 14: ends at ~85 epochs,\nvalidation loss levels off at ~0.035')

# Fig. 15: predicted vs simulated KER
fig, ax = plt.subplots(figsize=(6, 5.5))
parity_plot(ax, y[:, 1], oof_pred_ker, 'MLP-KER', '#8cc63f', 'residual kinetic energy (J)')
plt.tight_layout(); plt.show()

### A7 · Final models and predictions for new inputs
**What it does:** A5 and A6 give the *honest* accuracy (every sample is predicted by a model that never saw it).
Here one MLP per target is retrained on all 300 samples with the same settings, and `predict_sea_ker(T)` turns any thickness triple `[t1, t2, t3]` in mm into a SEA and KER prediction.
The last lines show three example designs.

> Stay inside the training ranges **t1 ∈ [2, 10], t2 ∈ [5, 15], t3 ∈ [10, 40] mm**: outside them the network extrapolates and is unreliable.

**Creates:** `final_models`, `predict_sea_ker`.

In [ ]:
# =====================================================================
# FINAL MLPs TRAINED ON ALL 300 SAMPLES + PREDICTION FOR NEW INPUTS
# =====================================================================
final_models = {}
for name, col in [('SEA', 0), ('KER', 1)]:
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = build_mlp(**MLP_PARAMS[name])
    train_mlp(model, X_scale, y_scale[:, col])
    final_models[name] = model

def predict_sea_ker(T):
    """MLP predictions of SEA (J/kg) and KER (J) for thickness rows T = [[t1, t2, t3], ...] in mm."""
    Ts = scaler_X.transform(np.atleast_2d(T))
    sea = to_physical(predict_scaled(final_models['SEA'], Ts), 0)
    ker = to_physical(predict_scaled(final_models['KER'], Ts), 1)
    return sea, np.maximum(ker, 0.0)         # residual kinetic energy cannot be negative

final_sea, final_ker = predict_sea_ker(X)
print(f"Final MLPs, R² on their own 300 training samples (a fit check, not a validation): "
      f"SEA {r2_score(y[:, 0], final_sea):.4f}, KER {r2_score(y[:, 1], final_ker):.4f}")
print("Honest accuracy = the 5-fold validation metrics of sections 5 and 6.")

# ---- Example: predict for new thickness combinations (mm). Stay inside the training ranges:
#      t1 in [2, 10], t2 in [5, 15], t3 in [10, 40]; outside them the MLP extrapolates and is unreliable.
new_inputs = np.array([[5.0, 10.0, 32.0],
                       [2.0, 14.95, 15.73],
                       [6.0, 9.0, 25.0]])
sea_new, ker_new = predict_sea_ker(new_inputs)
print(pd.DataFrame({'t1 (mm)': new_inputs[:, 0], 't2 (mm)': new_inputs[:, 1], 't3 (mm)': new_inputs[:, 2],
                    'SEA pred (J/kg)': sea_new.round(1), 'KER pred (J)': ker_new.round(1)}).to_string(index=False))

---
## Part B — SVM and Random Forest benchmark
*A separate study. Part A is complete without it and never uses anything created here.*

### B1 · What the benchmark does and how it connects to Part A
**Question (paper §4.3, Table 9):** does the MLP predict SEA and KER better than a support-vector regressor (SVM) and a random forest (RF)?

**How it is done**
- SVM (`SVR`, RBF kernel) and RF (`RandomForestRegressor`) are fitted to the **same scaled data** (`X_scale`, `y_scale`) and scored on the **same 5 outer folds** (`kf`) as the MLP, with the same metrics (`regression_metrics`) and the same conversion back to physical units (`to_physical`).
- The paper does not report SVM or RF hyperparameters, so they are tuned by **nested cross-validation**: on each outer training fold an inner 5-fold grid search picks the hyperparameters, and the outer validation fold is used only for scoring.
- The MLP is **not** re-tuned here: its scores are the A5 and A6 results (`mlp_sea`, `mlp_ker`), obtained with the paper's selected hyperparameters. Neither model family chooses its settings on the folds it is scored on.

**Needs from Part A:** `X`, `y`, `armor`, `X_scale`, `y_scale`, `kf`, `SEED`, `regression_metrics`, `to_physical`, `parity_plot`, and the results of **A5 and A6**.

### B2 · Imports, search grids and the nested-CV function
**What it does:** imports the two extra model classes, checks that Part A has been run, defines the hyperparameter grids searched by SVM and RF, and defines `run_sklearn_cv`
(nested 5-fold CV for any scikit-learn regressor). Nothing is fitted yet.

In [ ]:
# Extra imports used only by this benchmark (Part A does not need them)
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

# Part A must have been run first (A1-A3 for the data and helpers, A5 and A6 for the MLP scores)
for _name in ('armor', 'X_scale', 'y_scale', 'y', 'kf', 'regression_metrics', 'to_physical', 'parity_plot',
              'mlp_sea', 'mlp_ker', 'oof_pred_sea', 'oof_pred_ker'):
    assert _name in globals(), f"Run Part A first (A1-A3, A5, A6): '{_name}' is missing"

# =====================================================================
# SVM AND RF BENCHMARKS (NESTED 5-FOLD CV ON THE SAME OUTER FOLDS AS THE MLP)
# =====================================================================
SVR_GRID = {'C': [1, 10, 100, 1000], 'gamma': ['scale', 1, 2, 5], 'epsilon': [0.005, 0.01, 0.05]}
RF_GRID = {'n_estimators': [300], 'max_depth': [None, 10], 'min_samples_leaf': [1, 2, 4], 'max_features': [1.0, 0.67]}

def run_sklearn_cv(base_estimator, param_grid, col):
    """Nested 5-fold CV: an inner grid search on each outer training fold, scoring on the outer validation fold."""
    inner = KFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_metrics, chosen = [], []
    oof_pred = np.zeros(len(X_scale))
    for train_idx, test_idx in kf.split(X_scale):
        search = GridSearchCV(base_estimator, param_grid, cv=inner, scoring='neg_mean_squared_error', n_jobs=-1)
        search.fit(X_scale[train_idx], y_scale[train_idx, col])
        pred_raw = to_physical(search.predict(X_scale[test_idx]), col)
        train_raw = to_physical(search.predict(X_scale[train_idx]), col)
        m = regression_metrics(y[test_idx, col], pred_raw)
        m_train = regression_metrics(y[train_idx, col], train_raw)
        m.update(R2_train=m_train['R2'], RMSE_train=m_train['RMSE'])
        fold_metrics.append(m)
        chosen.append(search.best_params_)
        oof_pred[test_idx] = pred_raw
    return pd.DataFrame(fold_metrics).mean(), oof_pred, chosen

### B3 · Run SVM and RF (nested CV)
**What it does:** for SEA and KER, runs the nested CV for SVM and RF (the settings chosen on each outer fold are printed) and collects the MLP results from A5/A6 into the same dictionaries.

**Creates:** `bench[(model, target)]` (mean validation metrics) and `oof_bench[(model, target)]` (out-of-fold predictions) for the models MLP, SVM, RF and the targets SEA, KER.

In [ ]:
bench, oof_bench = {}, {}
for name, col in [('SEA', 0), ('KER', 1)]:
    bench[('SVM', name)], oof_bench[('SVM', name)], svr_chosen = run_sklearn_cv(SVR(kernel='rbf'), SVR_GRID, col)
    bench[('RF', name)], oof_bench[('RF', name)], rf_chosen = run_sklearn_cv(
        RandomForestRegressor(random_state=SEED), RF_GRID, col)
    print(f"{name}: SVR settings chosen per outer fold: {svr_chosen}")
    print(f"{name}: RF settings chosen per outer fold : {rf_chosen}\n")

bench[('MLP', 'SEA')], oof_bench[('MLP', 'SEA')] = mlp_sea, oof_pred_sea
bench[('MLP', 'KER')], oof_bench[('MLP', 'KER')] = mlp_ker, oof_pred_ker

### B4 · Table 9: MLP vs SVM vs RF, paper vs reproduced
**What it does:** builds the paper's Table 9 next to the reproduced numbers (mean of the 5 validation folds; the training columns show how much each model overfits), ranks the three models by R², MAE and RMSE
(the paper reports MLP > SVM > RF), and prints a consistency check: the target spread implied by the paper's own RMSE and R² (`RMSE / sqrt(1 − R²)`) compared with the dataset's actual standard deviation.

In [ ]:
# Table 9 of the paper
TABLE9 = {('MLP', 'SEA'): (35.53, 56.62, 0.901), ('SVM', 'SEA'): (44.25, 70.93, 0.87), ('RF', 'SEA'): (51.82, 84.70, 0.84),
          ('MLP', 'KER'): (109.66, 190.88, 0.95), ('SVM', 'KER'): (148.30, 223.39, 0.92), ('RF', 'KER'): (158.83, 242.52, 0.91)}
rows = []
for (model_name, target), (mae_p, rmse_p, r2_p) in TABLE9.items():
    r = bench[(model_name, target)]
    rows.append([model_name, target, mae_p, r['MAE'], rmse_p, r['RMSE'], r2_p, r['R2'], r['R2_train'], r['RMSE_train']])
table9 = pd.DataFrame(rows, columns=['Model', 'Target', 'MAE paper', 'MAE repr', 'RMSE paper', 'RMSE repr',
                                     'R2 paper', 'R2 repr', 'training R2', 'training RMSE'])
print("Table 9 - paper vs reproduced (mean of the 5 validation folds; training columns = mean of the training folds):")
print(table9.round(3).to_string(index=False))

print()
for target in ['SEA', 'KER']:
    for metric, best_is_high in [('R2', True), ('MAE', False), ('RMSE', False)]:
        order = sorted(['MLP', 'SVM', 'RF'], key=lambda m: bench[(m, target)][metric], reverse=best_is_high)
        print(f"{target} ranking by {metric:4s}: {' > '.join(order)}   (paper: MLP > SVM > RF)")

# Internal consistency of Table 9: R2 = 1 - RMSE^2 / var(y) implies the std of the validation targets
print("\nTarget std implied by the paper's RMSE and R2 (dataset std: SEA %.1f J/kg, KER %.1f J):"
      % (armor['SEA'].std(), armor['KER'].std()))
for (model_name, target), (mae_p, rmse_p, r2_p) in TABLE9.items():
    print(f"  {model_name}-{target}: {rmse_p / np.sqrt(1 - r2_p):.1f}")

### B5 · Predicted-vs-simulated plots for all three models
**What it does:** reproduces Fig. 16, a 2×3 grid of parity plots (rows SEA, KER; columns MLP, SVM, RF). The closer the points lie to the red diagonal, the better the model; each panel shows the R² of all 300 out-of-fold predictions.

In [ ]:
# Fig. 16: predicted vs simulated for the three models
colors = {'MLP': '#8cc63f', 'SVM': '#29abe2', 'RF': '#fbb03b'}
units = {'SEA': (0, 'specific energy absorption (J/kg)'), 'KER': (1, 'residual kinetic energy (J)')}
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
for i, target in enumerate(['SEA', 'KER']):
    col, unit_name = units[target]
    for j, model_name in enumerate(['MLP', 'SVM', 'RF']):
        parity_plot(axes[i, j], y[:, col], oof_bench[(model_name, target)], f'{model_name}-{target}',
                    colors[model_name], unit_name)
fig.suptitle('Fig. 16 reproduced', fontsize=15)
plt.tight_layout(); plt.show()